In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Similitud de contenidos programáticos

Este análisis tiene como objetivo identificar similitud entre contenidos de asignaturas utilizando técnicas de procesamiento de lenguaje natural (NLP), con el fin de apoyar decisiones de asignación docente y homologación académica.

### Metodología

Se utiliza un enfoque basado en TF-IDF para representar los textos y similitud del coseno para medir la relación entre los contenidos de las asignaturas.

Este método permite identificar qué tan similares son dos materias a partir de sus descripciones.

#### Materias relacionadas con Estadistica

Carga los contenidos programaticos de cada materia para ser comparados.

In [ ]:
from pathlib import Path

BASE_DIR = Path().resolve()
DATA_DIR = BASE_DIR / "data"

In [3]:
ruta_archivo = DATA_DIR / "Estadistica.xlsx" 
nombres_bases = {
    "Estadistica1": "Estadistica1",
    "Estadistica": "Estadistica",
    "Estadistica2": "Estadistica2",
    "Probabilidad y estadistica": "Probabilidad y estadistica",
    "Probabilidad": "Probabilidad"
}


hojas = pd.read_excel(ruta_archivo, sheet_name=None)

#seleccionar solo las hojas que están en nombres_bases
bases_renombradas = {nuevo_nombre: hojas[hoja] for hoja, nuevo_nombre in nombres_bases.items() if hoja in hojas}

In [4]:
estadistica1 = bases_renombradas["Estadistica1"]
estadistica = bases_renombradas["Estadistica"]
estadistica2 = bases_renombradas["Estadistica2"]
probabilidad_estadistica = bases_renombradas["Probabilidad y estadistica"]
probabilidad= bases_renombradas["Probabilidad"]

In [15]:
# Lista de bases y nombres
bases = [estadistica1, estadistica, estadistica2, probabilidad_estadistica, probabilidad]
nombres_bases = ["estadistica1", "estadistica", "estadistica2", "probabilidad_estadistica", "probabilidad"]

# Concatenar todo el contenido de cada base en una sola cadena de texto
contenidos = {nombre: " ".join(base["Contenido"].astype(str)) for nombre, base in zip(nombres_bases, bases)}

# Crear un DataFrame con los contenidos concatenados
combinada = pd.DataFrame(list(contenidos.items()), columns=['Base', 'Contenido'])

# Vectorizar los contenidos concatenados
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(combinada['Contenido'])

# Calcular la similitud coseno entre las bases
similitudes = cosine_similarity(tfidf_matrix)

# Crear un DataFrame para almacenar los resultados
resultados = []
for i in range(len(nombres_bases)):
    for j in range(i + 1, len(nombres_bases)):
        base_actual = nombres_bases[i]
        base_comparada = nombres_bases[j]
        similitud = similitudes[i, j]
        resultados.append({
            "Base Actual": base_actual,
            "Base Comparada": base_comparada,
            "Similitud (%)": round(similitud * 100, 2)
        })

# Convertir los resultados a un DataFrame
resultados = pd.DataFrame(resultados)







In [16]:
def comparar_bases(bases, nombres_bases, umbral_alto=75, umbral_medio=60):
    contenidos = {
        nombre: " ".join(base["Contenido"].astype(str)) 
        for nombre, base in zip(nombres_bases, bases)
    }
    
    combinada = pd.DataFrame(
        list(contenidos.items()), 
        columns=["Base", "Contenido"]
    )

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(combinada["Contenido"])

    similitudes = cosine_similarity(matrix)

    resultados = []

    for i in range(len(nombres_bases)):
        for j in range(i + 1, len(nombres_bases)):
            base_actual = nombres_bases[i]
            base_comparada = nombres_bases[j]
            similitud = round(similitudes[i, j] * 100, 2)

            if similitud >= umbral_alto:
                nivel = "Alta"
                interpretacion = f"{base_actual} ≈ {base_comparada} (Alta similitud)"
            elif similitud >= umbral_medio:
                nivel = "Media"
                interpretacion = f"{base_actual} ~ {base_comparada} (Similitud media)"
            else:
                nivel = "Baja"
                interpretacion = f"{base_actual} ≠ {base_comparada} (Baja similitud)"

            resultados.append({
                "Base Actual": base_actual,
                "Base Comparada": base_comparada,
                "Similitud (%)": similitud,
                "Nivel de similitud": nivel,
                "Interpretación": interpretacion
            })

    return pd.DataFrame(resultados)

## Resultados

Se presentan los niveles de similitud entre asignaturas, clasificados en alta, media y baja similitud.

In [23]:
similitud_df

,Base Actual,Base Comparada,Similitud (%),Interpretacion
0,estadistica1,estadistica,98.01,Alta similitud
1,estadistica1,estadistica2,58.12,Baja similitud
2,estadistica1,probabilidad_estadistica,65.04,Similitud media
3,estadistica1,probabilidad,75.04,Alta similitud
4,estadistica,estadistica2,58.60,Baja similitud
5,estadistica,probabilidad_estadistica,67.96,Similitud media
6,estadistica,probabilidad,78.58,Alta similitud
7,estadistica2,probabilidad_estadistica,82.41,Alta similitud
8,estadistica2,probabilidad,46.82,Baja similitud
9,probabilidad_estadistica,probabilidad,63.05,Similitud media


## Interpretación

Los resultados permiten identificar asignaturas con alta similitud en sus contenidos, lo que sugiere posibles equivalencias académicas.

Las similitudes medias indican relación parcial, mientras que las bajas reflejan diferencias significativas entre los contenidos.

## Conclusiones

El modelo permite automatizar la comparación de contenidos programáticos, apoyando la toma de decisiones en procesos de asignación docente y homologación de materias.

Este enfoque puede escalarse a otras áreas donde sea necesario identificar similitud entre textos.
